In [ ]:
#base environment for python scripts

In [ ]:
import sys
import os
import argparse
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

In [ ]:
#global variables

In [ ]:
NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]

In [ ]:
#load peaks and hits

In [ ]:
X_peaks_coords = pd.read_csv("~/Treeshrew/FileSave/TreeShrew_peak_1000bp_OBO/modified_peak/C3.2_W1000bp_Peak.modified.bed", sep='\t', header=None, names = NARROWPEAK_SCHEMA)
X_peaks_coords.head()

In [ ]:
X_peaks_coords.shape

In [ ]:
X_peaks_coords["1"]= X_peaks_coords["chr"] + ":" + (X_peaks_coords["start"]+1).astype(str) + "-" + X_peaks_coords["end"].astype(str)
X_peaks_coords.head()

In [ ]:
X_peaks_coords["2"]=X_peaks_coords.index
X_peaks_coords.head()

In [ ]:
hits = pd.read_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/finemo/counts/hit_call/hits.tsv", sep="\t")
hits.head()

In [ ]:
hits = hits[hits['peak_id'].isin(X_peaks_coords.index)]
max(hits['peak_id'])

In [ ]:
motif_matrix = pd.crosstab(hits['peak_id'], hits['motif_name'])
motif_matrix.head()

In [ ]:
motif_matrix.sum(axis=1).describe()

In [ ]:
column_sums = motif_matrix.sum()
column_sums
# 按总和降序排序
sorted_columns = column_sums.sort_values(ascending=False)
sorted_columns
top_6_columns = sorted_columns.head(6).index.tolist()
top_6_columns

In [ ]:
motif_matrix.to_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/finemo/counts/TS_C3.2_Allpeaks_motif_presence_matrix.csv")

In [ ]:
#Create Binary Matrix for Co-occurrence (Present or Not)二值矩阵
motif_binary = (motif_matrix > 0).astype(int)

In [ ]:
# Calculate Co-occurrence Matrix共现矩阵
# (Transpose) dot (Matrix) -> (Motifs x Peaks) dot (Peaks x Motifs) -> (Motifs x Motifs)
cooccurrence_matrix = motif_binary.T.dot(motif_binary)

In [ ]:
cooccurrence_matrix.to_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/finemo/counts/TS_C3.2_Allpeaks_motif_cooccurrence_matrix.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
#Plot Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cooccurrence_matrix, annot=True, fmt='d', cmap='Reds')
plt.title('Motif Co-occurrence Matrix (Number of Peaks shared)')
#plt.savefig('motif_cooccurrence.png')
plt.show()

In [ ]:
# --- 计算余弦相似度 ---

# 计算每个 Motif 出现的总次数 (提取对角线元素)。这相当于每个向量的模的平方 (|A|^2)
motif_sums = np.diag(cooccurrence_matrix) 

# 计算分母矩阵 (|A| * |B|)
# np.sqrt(motif_sums) 得到每个向量的模。np.outer(s, s) 做外积，得到 n×n 的分母矩阵。
denom = np.outer(np.sqrt(motif_sums), np.sqrt(motif_sums))

# 计算余弦相似度矩阵
# 避免分母为 0 的情况（如果有 Motif 一次都没出现）
# 用 with np.errstate 忽略除以 0 的警告，或者直接处理
cosine_sim_matrix = cooccurrence_matrix / denom

# 将 NaN (除以0产生) 替换为 0
cosine_sim_matrix = cosine_sim_matrix.fillna(0)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(cosine_sim_matrix, 
            annot=False, # 相似度通常数值较多，建议关掉数值或只显示一部分
            cmap='viridis', # 或 'coolwarm', 'Reds'
            vmin=0, vmax=1) # 余弦相似度范围通常是 0 到 1 (对于非负数据)
plt.title('Motif Co-occurrence (Cosine Similarity)')
plt.show()